In [1]:
from utils import ImmutableState, State, Action, board_status, get_local_board_status, get_all_valid_actions, is_terminal, change_state, terminal_utility, invert, load_data
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

class UTTTModel(nn.Module):
    def __init__(self):
        super(UTTTModel, self).__init__()
        self.conv1 = nn.Conv2d(4, 16, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1)
        self.conv4 = nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1)
        self.fc1 = nn.Linear(32 * 9 * 9, 192)  # 32 channels from last conv layer
        #self.fc2 = nn.Linear(192, 256) #168 #224
        self.fc3 = nn.Linear(192, 128)
        self.fc4 = nn.Linear(128, 64)
        self.fc5 = nn.Linear(64, 1)
        #self.relu = nn.ReLU()
        #self.prelu = nn.PReLU()
        self.lrelu = nn.LeakyReLU(0.01)
    
    def forward(self, x):
        x = x.view(-1, 4, 9, 9)  # Reshape to 9x9 grid
        x = self.lrelu(self.conv1(x))
        x = self.lrelu(self.conv2(x))
        x = self.lrelu(self.conv3(x))
        x = self.lrelu(self.conv4(x))
        x = x.view(x.size(0), -1)  # Flatten for FC layers
        x = self.lrelu(self.fc1(x))
        #x = self.lrelu(self.fc2(x))
        x = self.lrelu(self.fc3(x))
        x = self.lrelu(self.fc4(x))
        #return torch.sigmoid(self.fc5(x)) #output range (0,1)
        return torch.tanh(self.fc5(x))  # Output score in range (-1, 1)

model2=UTTTModel()
print(model2)
print(sum(p.numel() for p in model2.parameters()))
from torch.utils.data import DataLoader, random_split

class UTTTDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        state, utility = self.data[idx]
        state_tensor = state_to_tensor(state)  # Convert state to tensor
        utility_tensor = torch.tensor([utility], dtype=torch.float32)  # Keep utility as tensor
        return state_tensor, utility_tensor

#wld be good to include in utils.py
def rotate_state(state, k):
    """
    Rotates a State object by 90° counterclockwise `k` times.
    - Rotates `board` (3x3x3x3)
    - Rotates `local_board_status` (3x3)
    - Adjusts `prev_local_action` accordingly
    - Keeps `fill_num` unchanged
    """
    board = state.board
    local_board_status = state.local_board_status
    prev_local_action = state.prev_local_action
    
    # Rotate metaboard (global 3x3 structure)
    rotated_metaboard = np.rot90(board, k, axes=(0, 1))
    
    # Rotate each localboard (3x3 grid inside each metaboard cell)
    rotated_board = np.zeros_like(board)
    for i in range(3):
        for j in range(3):
            rotated_board[i, j] = np.rot90(rotated_metaboard[i, j], k)

    # Rotate local board status
    rotated_local_board_status = np.rot90(local_board_status, k)

    # Rotate previous action
    if prev_local_action is not None:
        prev_row, prev_col = prev_local_action  # Previous move in (row, col) format
        new_row, new_col = rotate_coords(prev_row, prev_col, k)
        prev_local_action = (new_row, new_col)

    # Return a new State with the rotated attributes
    return ImmutableState(
        board=rotated_board,
        local_board_status=rotated_local_board_status,
        prev_local_action=prev_local_action,
        fill_num=state.fill_num  # Keep the fill number unchanged
    )

def rotate_coords(row, col, k):
    """
    Rotates a (row, col) coordinate within a 3x3 grid counterclockwise by 90° * k.
    """
    for _ in range(k % 4):  # Modulo 4 ensures valid rotations
        row, col = 2 - col, row  # Rotate 90° counterclockwise
    return row, col

def augment_data_with_rotations(data):
    """Generate rotated versions of states to enforce rotation invariance."""
    augmented_data = []
    
    for state, value in tqdm(data):        
        for k in range(4):  # Rotate 0, 90, 180, 270 degrees
            rotated_state = rotate_state(state, k)
            augmented_data.append((rotated_state, value))

    return augmented_data

def augment_data_with_inversion(data):
    return [(invert(state),-v) for state,v in tqdm(data)] #note invert() does not set local_board_status

def flip_coords_horizontal(row, col):
    return row, 2 - col

def flip_coords_vertical(row, col):
    return 2 - row, col

def flip_coords_diagonal_main(row, col):
    return col, row

def flip_coords_diagonal_anti(row, col):
    return 2 - col, 2 - row

def flip_state_horizontal(state):
    board = state.board.copy()
    flipped_board = np.zeros_like(board)

    for MR in range(3):
        for MC in range(3):
            for r in range(3):
                for c in range(3):
                    new_MC = 2 - MC
                    new_c = 2 - c
                    flipped_board[MR][new_MC][r][new_c] = board[MR][MC][r][c]

    flipped_local_status = np.fliplr(state.local_board_status)

    if state.prev_local_action is not None:
        r, c = state.prev_local_action
        flipped_action = flip_coords_horizontal(r, c)
    else:
        flipped_action = None

    return ImmutableState(
        board=flipped_board,
        local_board_status=flipped_local_status,
        prev_local_action=flipped_action,
        fill_num=state.fill_num
    )


def flip_state_vertical(state):
    board = state.board.copy()
    flipped_board = np.zeros_like(board)

    for MR in range(3):
        for MC in range(3):
            for r in range(3):
                for c in range(3):
                    new_MR = 2 - MR
                    new_r = 2 - r
                    flipped_board[new_MR][MC][new_r][c] = board[MR][MC][r][c]

    flipped_local_status = np.flipud(state.local_board_status)

    if state.prev_local_action is not None:
        r, c = state.prev_local_action
        flipped_action = flip_coords_vertical(r, c)
    else:
        flipped_action = None

    return ImmutableState(
        board=flipped_board,
        local_board_status=flipped_local_status,
        prev_local_action=flipped_action,
        fill_num=state.fill_num
    )


def flip_state_diagonal_main(state):
    board = state.board.copy()
    flipped_board = np.zeros_like(board)

    for MR in range(3):
        for MC in range(3):
            for r in range(3):
                for c in range(3):
                    new_MR = MC
                    new_MC = MR
                    new_r = c
                    new_c = r
                    flipped_board[new_MR][new_MC][new_r][new_c] = board[MR][MC][r][c]

    flipped_local_status = np.transpose(state.local_board_status)

    if state.prev_local_action is not None:
        r, c = state.prev_local_action
        flipped_action = flip_coords_diagonal_main(r, c)
    else:
        flipped_action = None

    return ImmutableState(
        board=flipped_board,
        local_board_status=flipped_local_status,
        prev_local_action=flipped_action,
        fill_num=state.fill_num
    )


def flip_state_diagonal_anti(state):
    board = state.board.copy()
    flipped_board = np.zeros_like(board)

    for MR in range(3):
        for MC in range(3):
            for r in range(3):
                for c in range(3):
                    new_MR = 2 - MC
                    new_MC = 2 - MR
                    new_r = 2 - c
                    new_c = 2 - r
                    flipped_board[new_MR][new_MC][new_r][new_c] = board[MR][MC][r][c]

    flipped_local_status = np.fliplr(np.flipud(np.transpose(state.local_board_status)))

    if state.prev_local_action is not None:
        r, c = state.prev_local_action
        flipped_action = flip_coords_diagonal_anti(r, c)
    else:
        flipped_action = None

    return ImmutableState(
        board=flipped_board,
        local_board_status=flipped_local_status,
        prev_local_action=flipped_action,
        fill_num=state.fill_num
    )


def augment_data_with_flips(data):
    augmented_data = []

    for state, value in tqdm(data):
        augmented_data.append((flip_state_horizontal(state), value))
        augmented_data.append((flip_state_vertical(state), value))
        augmented_data.append((flip_state_diagonal_main(state), value))
        augmented_data.append((flip_state_diagonal_anti(state), value))
    
    return augmented_data
    

def state_to_tensor(state):
    """
    Convert a 3x3x3x3 Ultimate Tic-Tac-Toe board state into a 4x9x9 tensor for the neural network.
    """
    board_3x3x3x3 = state.board.copy()

    # Convert 3x3x3x3 nested board into 9x9
    board_9x9 = np.zeros((9, 9), dtype=np.float32)
    
    for meta_row in range(3):
        for meta_col in range(3):
            for local_row in range(3):
                for local_col in range(3):
                    global_row = meta_row * 3 + local_row
                    global_col = meta_col * 3 + local_col
                    board_9x9[global_row][global_col] = board_3x3x3x3[meta_row][meta_col][local_row][local_col]

    # Normalize values: 0 stays 0, AI (1) stays 1, Opponent (2) becomes -1
    board_9x9[board_9x9 == 2] = -1

    # Convert to PyTorch tensor and add batch dimension
    board_tensor = torch.tensor(board_9x9, dtype=torch.float32).unsqueeze(0)  # Shape: (1, 9, 9)
    
    turn_tensor = torch.full((1,9,9), 1 if state.fill_num==1 else -1, dtype=torch.float32)
    
    action_9x9 = np.zeros((9,9), dtype=np.float32)
    valid_actions = get_all_valid_actions(state)
    for meta_row, meta_col, local_row, local_col in valid_actions:
        global_row = meta_row * 3 + local_row
        global_col = meta_col * 3 + local_col
        action_9x9[global_row][global_col] = 1
    
    action_tensor = torch.tensor(action_9x9, dtype=torch.float32).unsqueeze(0)
    
    outcome_9x9=np.zeros((9, 9), dtype=np.float32)
    lbs = get_local_board_status(board_3x3x3x3)  # 3x3 array
    for i in range(3):  # iterate over meta board rows
        for j in range(3):  # iterate over meta board cols
            local_status = lbs[i, j]
            if local_status == 1:
                fill_value = 1.0
            elif local_status == 2:
                fill_value = -1.0
            else:
                fill_value = 0.0

            # Fill the corresponding 3x3 block in the global 9x9 board
            row_start, row_end = i * 3, (i + 1) * 3
            col_start, col_end = j * 3, (j + 1) * 3
            outcome_9x9[row_start:row_end, col_start:col_end] = fill_value

    outcome_tensor = torch.tensor(outcome_9x9, dtype=torch.float32).unsqueeze(0)
    
    return torch.cat([turn_tensor,board_tensor,outcome_tensor,action_tensor],dim=0)


def evaluation(state, model):
    """
    Evaluates the board using the trained neural network.
    """
    if state.is_terminal():
        return 2*state.terminal_utility()-1
    state_tensor = state_to_tensor(state).unsqueeze(0)  # Add batch dimension
    with torch.no_grad():
        return model(state_tensor).item()  # Get NN evaluation score
    
def minimax(model, state, depth, alpha, beta, maximizing=True):
    if depth == 0 or state.is_terminal():
        return evaluation(state, model), None
    
    best_action = None
    
    if maximizing:
        max_eval = -float("inf")
        for action in state.get_all_valid_actions():
            new_state = state.change_state(action)
            eval, _ = minimax(model, new_state, depth - 1, alpha, beta, False)
            if eval > max_eval:
                max_eval = eval
                best_action = action
            alpha = max(alpha, eval)
            if beta <= alpha:
                break
        return max_eval, best_action
    else:
        min_eval = float("inf")
        for action in state.get_all_valid_actions():
            new_state = state.change_state(action)
            eval, _ = minimax(model, new_state, depth - 1, alpha, beta, True)
            if eval < min_eval:
                min_eval = eval
                best_action = action
            beta = min(beta, eval)
            if beta <= alpha:
                break
        return min_eval, best_action

torch.manual_seed(42)
np.random.seed(42)
def train_network(model, dataset, epochs=10, batch_size=64, lr=0.0001, val_split=0.1, test_split=0.1):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=0.0001)
    #loss_fn = nn.MSELoss()
    loss_fn = nn.SmoothL1Loss()
    #loss_fn = nn.BCELoss()
    
    dataset_size = len(dataset)
    test_size = int(test_split * dataset_size)
    val_size = int(val_split * dataset_size)
    train_size = dataset_size - val_size - test_size
    
    train_set, val_set, test_set = random_split(dataset, [train_size, val_size, test_size])

    #dataloader = torch.utils.data.DataLoader(dataset, num_workers=4, batch_size=batch_size, shuffle=True)
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=4)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=4)

    train_losses, val_losses = [], []
    for epoch in range(epochs):
        train_loss=0.0
        model.train()
        print(f"Starting epoch {epoch + 1}:")
        for state_tensor, target_score in tqdm(train_loader):
            optimizer.zero_grad(set_to_none=True)
            prediction = model(state_tensor)
            loss = loss_fn(prediction.squeeze(), target_score.squeeze())
            loss.backward()
            optimizer.step()
            train_loss+=loss.item()
        train_loss/=len(train_loader)
        train_losses.append(train_loss)
        print(f"Epoch {epoch + 1}: Training Loss = {train_loss:.8f}")
        
        
        model.eval()  # Set model to evaluation mode
        val_loss = 0.0
        with torch.no_grad():  # Disable gradient calculation
            for state_tensor, target_score in tqdm(val_loader):
                prediction = model(state_tensor)
                loss = loss_fn(prediction.squeeze(), target_score.squeeze())
                val_loss += loss.item()

        val_loss /= len(val_loader)
        val_losses.append(val_loss)
        print(f"Epoch {epoch + 1}: Validation Loss = {val_loss:.8f}")
    return train_loader,val_loader,test_loader

def evaluate_model(model, test_loader):
    model.eval()
    loss_fn = nn.SmoothL1Loss()
    test_loss = 0.0

    with torch.no_grad():
        for state_tensor, target_score in test_loader:
            prediction = model(state_tensor)
            loss = loss_fn(prediction.squeeze(), target_score.squeeze())
            test_loss += loss.item()

    test_loss /= len(test_loader)
    print(f"Final Test Loss: {test_loss:.8f}")
    return test_loss

data=load_data()
dataset = UTTTDataset(data)
dataloader = torch.utils.data.DataLoader(dataset, num_workers=4, batch_size=64, shuffle=True)

UTTTModel(
  (conv1): Conv2d(4, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=2592, out_features=192, bias=True)
  (fc3): Linear(in_features=192, out_features=128, bias=True)
  (fc4): Linear(in_features=128, out_features=64, bias=True)
  (fc5): Linear(in_features=64, out_features=1, bias=True)
  (lrelu): LeakyReLU(negative_slope=0.01)
)
554609


In [5]:
def evaluate_model(model, test_loader, threshold=0.1):
    model.eval()
    loss_fn = nn.SmoothL1Loss()
    test_loss = 0.0
    total = 0
    correct = 0

    with torch.no_grad():
        for state_tensor, target_score in tqdm(test_loader):
            prediction = model(state_tensor).squeeze()
            target_score = target_score.squeeze()

            loss = loss_fn(prediction, target_score)
            test_loss += loss.item()

            # Count accuracy as predictions close to ground truth
            correct += ((prediction - target_score).abs() < threshold).sum().item()
            total += prediction.numel()

    test_loss /= len(test_loader)
    accuracy = correct / total
    print(f"Final Test Loss: {test_loss:.8f}")
    print(f"Accuracy (within ±{threshold}): {accuracy * 100:.4f}%")
    return test_loss, accuracy

model2=UTTTModel()
model2.load_state_dict(torch.load("model_weights_2_20.pth", weights_only=False))
evaluate_model(model2, dataloader)
#maybe should train more

100%|███████████████████████████████████████████████████████████████████████████████| 1250/1250 [00:12<00:00, 98.24it/s]

Final Test Loss: 0.03702440
Accuracy (within ±0.1): 57.7287%


(0.037024402885884045, 0.5772875)